# Porto Taxi — Popular Routes, Activity Zones & Anomalies (Colab Enterprise demo)

This notebook renders the results of the Spark pipeline on an interactive map.
It reads the small **result CSVs** produced by the mining jobs (top routes for
Methods A/B/C, activity zones, anomalies) — NOT the raw 1.9 GB data — so it runs
instantly in Colab.

Pipeline that produced these files (run on DataProc, see `docs/DATAPROC.md`):
`clean_data → feature_engineering → spatial_encoding → route_mining_{maximal,
clustering,graph} → anomaly_analysis`. Outputs land in `gs://<bucket>/outputs/routes/`.

In [ ]:
# 1. Dependencies (Colab has pandas; add folium + h3)
!pip -q install folium==0.16.0 h3==3.7.7 gcsfs
import pandas as pd, folium, h3

In [ ]:
# 2. Where the result CSVs live. Use a gs:// bucket (Colab Enterprise auths to
#    GCS automatically) OR upload the CSVs and use a local dir.
BASE = 'gs://YOUR_BUCKET/outputs/routes'   # <-- edit; or './' if uploaded
SUFFIX = 'full'                            # 'full' on DataProc, 'sample' locally
def load(name):
    try:
        return pd.read_csv(f'{BASE}/{name}_{SUFFIX}.csv')
    except Exception as e:
        print('skip', name, e); return pd.DataFrame()
B = load('maximal_frequent_top100')
C = load('graph_heavy_paths_top100')
A = load('clustering_top100')
Z = load('activity_zones')
AN = load('anomalies_top50')
print('rows:', {'A':len(A),'B':len(B),'C':len(C),'zones':len(Z),'anom':len(AN)})

In [ ]:
# 3. Build the interactive map (top-N routes per method + zones + anomalies)
DELIM = '>'
def line(cells):
    return [list(h3.h3_to_geo(c)) for c in str(cells).split(DELIM) if c]
m = folium.Map(location=(41.157,-8.629), zoom_start=13, tiles='cartodbpositron')
def routes(df, col, pop, color, name, top=25, L=3, show=False):
    fg = folium.FeatureGroup(name=name, show=show)
    for _,r in df[df.get('min_len_km',0)==L].head(top).iterrows():
        pts = line(r[col])
        if len(pts)>=2:
            folium.PolyLine(pts, color=color, weight=3, opacity=0.7,
                            tooltip=f'{name} pop={r[pop]}').add_to(fg)
    fg.add_to(m)
if len(B): routes(B,'subroute','support','#1f5fbf','B maximal-frequent', show=True)
if len(C): routes(C,'route','support','#d1341c','C transition-graph')
if len(A): routes(A,'rep_route','cluster_size','#2e8b3d','A clustering', L=5)
if len(Z):
    zfg=folium.FeatureGroup(name='Activity zones', show=True)
    for _,z in Z.iterrows():
        folium.CircleMarker((z.lat,z.lon), radius=max(3,14-int(z['rank'])//4),
                            color='#e8850c', fill=True, fill_opacity=0.5,
                            tooltip=f"zone #{z['rank']}").add_to(zfg)
    zfg.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m

In [ ]:
# 4. Top-100 popular long sub-routes table (Method B) for a chosen length config
L = 3   # one of 1,3,5,10,20,40 km
B[B.min_len_km==L][['rank','support','length_km','n_cells']].head(100)